### Preprocessing Pipeline
This notebook implements the full preprocessing pipeline for the PhysionetMI dataset.
All parameters were decided during exploration in 00_data_exploration.ipynb.

### Pipeline order
1. Notch filter 60 Hz (powerline artifact, confirmed in all subjects)
2. Bandpass filter 8 to 30 Hz (mu and beta bands for motor imagery)
3. Common average re-referencing (best lateral separation vs linked mastoid and laplacian)
4. ICA EOG only removal (extended infomax, 20 components, Fp1/Fp2 proxy, threshold 3.0)
5. Epoch extraction (T1 left fist, T2 right fist, 0 to 3 seconds)
6. Hybrid threshold rejection (250 uV absolute ceiling or 3x subject median)
7. Baseline correction
8. Channel selection (C3, Cz, C4, FC3, FCz, CP3, CPz, CP4, FC4 excluded)
9. Z-score normalisation per channel

### Dataset
- 106 valid subjects (S088, S092, S100 excluded, 128 Hz sampling rate)
- Runs R04, R08, R12 per subject
- Binary classification: T1 left fist vs T2 right fist

### Initial Setup

In [21]:
import mne
from mne.preprocessing import ICA

import torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import random
import re

from tqdm import tqdm
import os

mne.set_log_level("WARNING")

# Reproducibility
np.random.seed(25)
random.seed(25)
torch.manual_seed(25)

# Paths
data_path = Path("../Data/raw/physionet_mi_data/eeg-motor-movementimagery-dataset")
processed_path = Path("../Data/processed")

In [ ]:
# dataset parameters
# runs for binary classification (left vs right hand)
# for multiclass: extend to include R06,R10,R14 for feet and both fists
runs = ['R04', 'R08', 'R12']
event_id = {'T1': 2, 'T2': 3}
n_samples = 480
sfreq = 160.0
tmax = (n_samples - 1) / sfreq
tmin = 0.0

# preprocessing parameters decided during exploration
notch_freq = 60.0
bandpass_low = 8.0
bandpass_high = 30.0
ica_broad_low = 1.0
ica_broad_high = 79.0
ica_n_components = 20
ica_eog_threshold = 3.0

artifact_relative_multiplier = 3.0

# channel selection decided during exploration
# FC4 excluded due to incorrect lateralisation direction
selected_channels = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4']

# subjects
all_subjects = [f"S{i:03d}" for i in range(1, 110)]
excluded = ['S088', 'S092', 'S100']
valid_subjects = [s for s in all_subjects if s not in excluded]

In [4]:
print(f"MNE version: {mne.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Data path exists: {data_path.exists()}")
print(f"Processed path exists: {processed_path.exists()}")
print(f"Valid subjects: {len(valid_subjects)}")
print(f"Selected channels: {selected_channels}")

MNE version: 1.12.1
PyTorch version: 2.11.0
Data path exists: True
Processed path exists: True
Valid subjects: 106
Selected channels: ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4']


### Channel renaming utility function

In [5]:
# reused across all preprocessing steps
# strips trailing dots and fixes capitalisation to match standard 1020 montage

montage = mne.channels.make_standard_montage('standard_1020')
montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

def rename_channels_standard(raw):
    # strip trailing dots first
    rename_map = {ch: ch.rstrip('.') for ch in raw.ch_names}
    raw.rename_channels(rename_map)

    # fix capitalisation to match standard montage
    cap_fix = {}
    for ch in raw.ch_names:
        upper = ch.upper()
        cap_fix[ch] = montage_lookup[upper] if upper in montage_lookup else ch
    raw.rename_channels(cap_fix)
    raw.set_montage(montage, on_missing='ignore')

    return raw


In [6]:
print("Channel renaming utility function defined")
print(f"Montage loaded: {len(montage.ch_names)} channels")

Channel renaming utility function defined
Montage loaded: 94 channels


### Single subject preprocessing function

In [7]:
def preprocess_subject(subject_id):
    """
    Full preprocessing pipeline for one subject.
    Returns clean epochs ready for model training.
    Steps: notch, bandpass, CAR, ICA EOG removal, epoching,
    hybrid threshold rejection, baseline correction,
    channel selection, z-score normalisation.
    """

    # load all three runs
    raw_list_narrow = []
    raw_list_broad = []

    for run in runs:
        edf_path = data_path / subject_id / f"{subject_id}{run}.edf"

        # narrow filtered version for analysis (8-30 Hz)
        raw_narrow = mne.io.read_raw_edf(edf_path, preload=True)
        raw_narrow.notch_filter(freqs=notch_freq)
        raw_narrow.filter(
            l_freq=bandpass_low,
            h_freq=bandpass_high,
            method='fir',
            fir_design='firwin'
        )
        raw_narrow.set_eeg_reference('average', projection=False)
        raw_list_narrow.append(raw_narrow)

        # broad filtered version for ICA fitting (1-79 Hz)
        raw_broad = mne.io.read_raw_edf(edf_path, preload=True)
        raw_broad.notch_filter(freqs=notch_freq)
        raw_broad.filter(
            l_freq=ica_broad_low,
            h_freq=ica_broad_high,
            method='fir',
            fir_design='firwin'
        )
        raw_broad.set_eeg_reference('average', projection=False)
        raw_list_broad.append(raw_broad)

    # concatenate runs
    raw_narrow_concat = mne.concatenate_raws(raw_list_narrow)
    raw_broad_concat = mne.concatenate_raws(raw_list_broad)

    # rename channels for both versions
    raw_narrow_concat = rename_channels_standard(raw_narrow_concat)
    raw_broad_concat = rename_channels_standard(raw_broad_concat)

    # fit ICA on broad filtered signal
    ica = ICA(
        n_components=ica_n_components,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_broad_concat)

    # find and remove EOG components using frontal channels as proxy
    eog_indices, _ = ica.find_bads_eog(
        raw_broad_concat,
        ch_name=['Fp1', 'Fp2'],
        threshold=ica_eog_threshold
    )
    ica.exclude = eog_indices

    # apply ICA to narrow filtered signal
    ica.apply(raw_narrow_concat)

    # extract epochs
    # binary classification: T1 left fist vs T2 right fist
    # for multiclass extension: add T3 (both fists) and T4 (both feet)
    # runs R06, R10, R14 contain feet imagery, R05, R09, R13 contain both fists
    # event_id would become {'T1': 2, 'T2': 3, 'T3': 2, 'T4': 3} on different runs
    # selected runs would also need updating from R04, R08, R12
    events, _ = mne.events_from_annotations(raw_narrow_concat)
    epochs = mne.Epochs(
        raw_narrow_concat,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        preload=True
    )

    # relative and IQR threshold rejection, artifact rejection using subject level thresholds
    """ a fixed amplitude limit of 250 uV was tested during exploration, but caused complete exclusion of high amplitude subjects as S009, S013, S024, S109 all had natural amplitudes above 250 uV, replaced with subject level relative and IQR thresholds. Where relative threshold (3x subject median) sets upper bound based on typical amplitude and IQR threshold (median + 3x IQR) catches outliers within subject's own distribution"""
    data = epochs.get_data()
    ptp = (data.max(axis=2) - data.min(axis=2)).max(axis=1) * 1e6
    subject_median = np.median(ptp)

    # relative threshold: 3x subject median
    relative_threshold = subject_median * artifact_relative_multiplier

    # IQR based outlier detection: median + 3x IQR
    q75 = np.percentile(ptp, 75)
    q25 = np.percentile(ptp, 25)
    iqr = q75 - q25
    iqr_threshold = subject_median + (3 * iqr)

    keep_mask = (ptp <= relative_threshold) & (ptp <= iqr_threshold)
    epochs = epochs[keep_mask]

    # baseline correction removed
    # bandpass filter at 8-30 Hz removes frequencies below 8 Hz
    # covering the same low frequency contamination baseline correction addresses
    # epochs.apply_baseline((0, 0.1))

    # channel selection
    epochs.pick_channels(selected_channels)

    # z-score normalisation per channel
    data = epochs.get_data()
    mean = data.mean(axis=(0, 2), keepdims=True)
    std = data.std(axis=(0, 2), keepdims=True)
    data_normalised = (data - mean) / (std + 1e-8)

    return data_normalised, epochs.events[:, 2], len(eog_indices), keep_mask.sum()

### Test on S001

In [8]:
# test preprocessing on one subject
# checking the output shapes, trail count, rejected trials, Eog components removed.

subject_id = 'S001'
data, labels, n_eog_removed, n_trials_kept = preprocess_subject(subject_id)

print(f'Subject: {subject_id}')
print(f'Data Shape: {data.shape} (trials x channels x timepoints)')
print(f"Labels: {np.unique(labels, return_counts=True)}")
print(f'EOG components removed: {n_eog_removed}')
print(f'Trials kept after rejection: {n_trials_kept}')
print(f"Expected timepoints: {n_samples} (actual: {data.shape[2]})")
print(f'Data mean: {data.mean():.4f} (should be close to 0)')
print(f'Data std: {data.std():.4f} (should be close to 1)')


Subject: S001
Data Shape: (44, 8, 480) (trials x channels x timepoints)
Labels: (array([2, 3]), array([22, 22]))
EOG components removed: 1
Trials kept after rejection: 44
Expected timepoints: 480 (actual: 480)
Data mean: -0.0000 (should be close to 0)
Data std: 0.9989 (should be close to 1)


### Save Data Function

In [9]:
save_path = processed_path / f"{subject_id}_preprocessed.npz"

def save_preprocessed(subject_id, data, labels, n_eog_removed, n_trials_kept):
    """
    saves preprocessed data for one subject as npz file
    data shape: trials x channels x timepoints
    labels: 2 for T1 left fist, 3 for T2 right fist
    """

    np.savez(
        save_path,
        data = data,
        labels = labels,
        n_eog_removed = n_eog_removed,
        n_trials_kept = n_trials_kept,
        selected_channels = selected_channels,
        sfreq = sfreq,
        n_samples = n_samples
    )

### run preprocessing pipeline on all 106 valid subjects

In [10]:

# saves one npz file per subject to data/processed directory
# skips already processed subjects to allow resuming if interrupted

failed_subjects = []
skipped_subjects = []
processed_subjects =[]

for subject_id in tqdm(valid_subjects, desc = "Preprocessing subjects"):
    save_path = processed_path / f"{subject_id}_processed.npz"

    # skip if already processed
    if save_path.exists():
        skipped_subjects.append(subject_id)
        continue
    try:
        data, labels, n_eog_removed, n_trials_kept = preprocess_subject(subject_id)
        save_preprocessed(subject_id, data, labels, n_eog_removed, n_trials_kept)
        processed_subjects.append(subject_id)

    except Exception as e:
        print(f"\n{subject_id} failed: {e}")
        failed_subjects.append(subject_id)
        continue

print(f"\nPreprocessing complete.")
print(f"Processed subjects: {len(processed_subjects)}")
print(f"Skipped subjects (already processed): {len(skipped_subjects)}")
print(f"Failed subjects: {len(failed_subjects)}")
if failed_subjects:
    print(f"Failed subject IDs: {failed_subjects}")


Preprocessing subjects:   0%|          | 0/106 [00:00<?, ?it/s]

Preprocessing subjects: 100%|██████████| 106/106 [00:00<00:00, 11501.95it/s]


Preprocessing complete.
Processed subjects: 0
Skipped subjects (already processed): 106
Failed subjects: 0


In [11]:
# verify processed files

processed_files = list(processed_path.glob('*.npz'))
print(f"Total files saved: {len(processed_files)}")

# load one file and check contents
sample = np.load(processed_path / 'S001_processed.npz')
print(f"\nS001 file contents:")
print(f"  data shape: {sample['data'].shape}")
print(f"  labels: {np.unique(sample['labels'], return_counts=True)}")
print(f"  n_eog_removed: {sample['n_eog_removed']}")
print(f"  n_trials_kept: {sample['n_trials_kept']}")
print(f"  selected_channels: {sample['selected_channels']}")
print(f"  sfreq: {sample['sfreq']}")
print(f"  n_samples: {sample['n_samples']}")

# check a few more subjects for consistency
print(f"\nSpot check across subjects:")
print(f"{'Subject':<10} {'Trials':<10} {'Shape':<20} {'Labels balanced':<20}")
print("-" * 62)
for subject_id in ['S001', 'S020', 'S050', 'S070', 'S090', 'S009', 'S013']:
    data = np.load(processed_path / f'{subject_id}_processed.npz')
    shape = data['data'].shape
    labels = data['labels']
    unique, counts = np.unique(labels, return_counts=True)
    balanced = abs(counts[0] - counts[1]) <= 3
    print(f"{subject_id:<10} {shape[0]:<10} {str(shape):<20} {str(balanced):<20}")

Total files saved: 106

S001 file contents:
  data shape: (44, 8, 480)
  labels: (array([2, 3]), array([22, 22]))
  n_eog_removed: 1
  n_trials_kept: 44
  selected_channels: ['C3' 'Cz' 'C4' 'FC3' 'FCz' 'CP3' 'CPz' 'CP4']
  sfreq: 160.0
  n_samples: 480

Spot check across subjects:
Subject    Trials     Shape                Labels balanced     
--------------------------------------------------------------
S001       44         (44, 8, 480)         True                
S020       43         (43, 8, 480)         True                
S050       43         (43, 8, 480)         True                
S070       44         (44, 8, 480)         True                
S090       45         (45, 8, 480)         True                
S009       45         (45, 8, 480)         True                
S013       45         (45, 8, 480)         True                


### investigate failed subjects
investigation cell, run before threshold change showed S009, S013, S024, S109 had natural amplitudes above 250 uV led to removal of absolute ceiling, see decision log 20 May

In [12]:
# # local variable for investigation only, removed from main pipeline
# artifact_threshold_uv = 250.0
# failed = ['S009','S013','S024','S109']

# for subject_id in failed:
#     raw_list = []
#     for run in ['R04', 'R08', 'R12']:
#         edf_path = data_path / subject_id / f"{subject_id}{run}.edf"
#         raw_s = mne.io.read_raw_edf(edf_path, preload=True)
#         raw_s.notch_filter(freqs = 60)
#         raw_s.filter(l_freq = bandpass_low, h_freq = bandpass_high, method = 'fir', fir_design = 'firwin')
#         raw_s.set_eeg_reference('average', projection=False)
#         raw_list.append(raw_s)

#     raw_concat = mne.concatenate_raws(raw_list)
#     raw_concat = rename_channels_standard(raw_concat)

#     events, _ = mne.events_from_annotations(raw_concat)
#     epochs = mne.Epochs(
#     raw_concat,
#     events,
#     event_id=event_id,
#     tmin=tmin,
#     tmax=tmax,
#     baseline=None,
#     preload=True
#     )

#     data = epochs.get_data()
#     ptp = (data.max(axis=2) - data.min(axis=2)).max(axis=1) * 1e6
#     subject_median = np.median(ptp)
#     relative_threshold = subject_median * artifact_relative_multiplier
#     keep_mask = (ptp <= artifact_threshold_uv) & (ptp <= relative_threshold)

#     print(f"\n{subject_id}:")
#     print(f"  Total trials: {len(ptp)}")
#     print(f"  Subject median ptp: {subject_median:.1f} uV")
#     print(f"  Relative threshold: {relative_threshold:.1f} uV")
#     print(f"  Absolute threshold: {artifact_threshold_uv} uV")
#     print(f"  Trials kept: {keep_mask.sum()}")
#     print(f"  Trial amplitudes: {ptp.round(1)}")

### testing IQR based threshold across diverse subject set

IQR Threshold Test
Testing IQR based threshold across 13 subjects after absolute ceiling failure.
Confirmed all four failed subjects now keep 100% of trials.
Led to threshold update in preprocessing function. See decision log 20 May.

In [13]:
# # testing IQR based threshold across diverse subject set
# # covering failed, noisy, normal, low amplitude, and unseen subjects

# test_subjects = [
#     'S009', 'S013', 'S024', 'S109',  # failed subjects
#     'S070', 'S090',                    # noisy from exploration
#     'S001', 'S020', 'S050',           # normal to low amplitude
#     'S030', 'S010',                    # medium amplitude
#     'S040', 'S060', 'S080'            # unseen subjects
# ]

# print(f"{'Subject':<10} {'Median uV':<12} {'IQR':<10} {'IQR thresh':<14} {'Rel thresh':<14} {'Kept':<8} {'Total':<8} {'Pct':<8}")
# print("-" * 86)

# for subject_id in test_subjects:
#     raw_list = []
#     for run in ['R04', 'R08', 'R12']:
#         edf_path = data_path / subject_id / f"{subject_id}{run}.edf"
#         raw_s = mne.io.read_raw_edf(edf_path, preload=True)
#         raw_s.notch_filter(freqs=60)
#         raw_s.filter(l_freq=bandpass_low, h_freq=bandpass_high,
#                     method='fir', fir_design='firwin')
#         raw_s.set_eeg_reference('average', projection=False)
#         raw_list.append(raw_s)

#     raw_concat = mne.concatenate_raws(raw_list)
#     raw_concat = rename_channels_standard(raw_concat)

#     events, _ = mne.events_from_annotations(raw_concat)
#     epochs = mne.Epochs(
#         raw_concat,
#         events,
#         event_id=event_id,
#         tmin=tmin,
#         tmax=tmax,
#         baseline=None,
#         preload=True
#     )

#     data = epochs.get_data()
#     ptp = (data.max(axis=2) - data.min(axis=2)).max(axis=1) * 1e6

#     subject_median = np.median(ptp)
#     q75 = np.percentile(ptp, 75)
#     q25 = np.percentile(ptp, 25)
#     iqr = q75 - q25
#     iqr_threshold = subject_median + (3 * iqr)
#     relative_threshold = subject_median * artifact_relative_multiplier

#     keep_mask = (ptp <= relative_threshold) & (ptp <= iqr_threshold)

#     print(
#         f"{subject_id:<10} {subject_median:<12.1f} {iqr:<10.1f} "
#         f"{iqr_threshold:<14.1f} {relative_threshold:<14.1f} "
#         f"{keep_mask.sum():<8} {len(ptp):<8} "
#         f"{keep_mask.sum()/len(ptp)*100:<8.1f}"
#     )

### check trial counts and label distribution across all 106 subjects


In [14]:
# final check across all subjects for trial counts and label balance after preprocessing, to identify any remaining issues before model training.
trial_counts = []
label_imbalances = []

for subject_id in valid_subjects:
    data = np.load(processed_path / f'{subject_id}_processed.npz')
    n_trials = data['data'].shape[0]
    labels = data['labels']
    unique, counts = np.unique(labels, return_counts=True)
    trial_counts.append(n_trials)
    if len(counts) == 2:
        imbalance = abs(counts[0] - counts[1])
    else:
        imbalance = n_trials
    label_imbalances.append(imbalance)

trial_counts = np.array(trial_counts)
print(f"Trial count distribution across 106 subjects:")
print(f"  min: {trial_counts.min()}")
print(f"  max: {trial_counts.max()}")
print(f"  mean: {trial_counts.mean():.1f}")
print(f"  median: {np.median(trial_counts):.1f}")
print(f"  std: {trial_counts.std():.1f}")
print(f"\nSubjects with fewer than 30 trials:")
for i, subject_id in enumerate(valid_subjects):
    if trial_counts[i] < 30:
        print(f"  {subject_id}: {trial_counts[i]} trials")

print(f"\nLabel imbalance distribution:")
print(f"  max imbalance: {max(label_imbalances)} trials")
print(f"  subjects with imbalance > 3: {sum(i > 3 for i in label_imbalances)}")

Trial count distribution across 106 subjects:
  min: 38
  max: 45
  mean: 44.2
  median: 45.0
  std: 1.1

Subjects with fewer than 30 trials:

Label imbalance distribution:
  max imbalance: 5 trials
  subjects with imbalance > 3: 6


### label verification across all valid subjects

In [15]:
# confirming T1=left fist, T2=right fist for all three runs
# R04 verified during exploration, R08 and R12 not yet checked
# verifying binary classification runs only
# for multiclass extension: add R05, R06, R09, R10, R13, R14 and verify separately

runs_to_check = ['R04', 'R08', 'R12']
label_issues = []

print(f"Checking label mapping across all 106 subjects and 3 runs...")
print(f"Expected: T1=2 (left fist), T2=3 (right fist)\n")

# check a sample of subjects for each run
# checking all 106 x 3 = 318 files would take too long
# instead check 10 subjects per run spread across the dataset
sample_subjects = valid_subjects[::10][:10]

for run in runs_to_check:
    print(f"Run {run}:")
    run_issues = []

    for subject_id in sample_subjects:
        edf_path = data_path / subject_id / f"{subject_id}{run}.edf"
        raw_check = mne.io.read_raw_edf(edf_path, preload=False)
        events, event_id_found = mne.events_from_annotations(raw_check)

        # check if T1 and T2 are present with correct codes
        if 'T1' not in event_id_found or 'T2' not in event_id_found:
            run_issues.append(f"{subject_id}: missing T1 or T2")
            continue

        t1_code = event_id_found['T1']
        t2_code = event_id_found['T2']

        if t1_code != 2 or t2_code != 3:
            run_issues.append(
                f"{subject_id}: unexpected codes T1={t1_code} T2={t2_code}"
            )

    if run_issues:
        print(f"  Issues found:")
        for issue in run_issues:
            print(f"    {issue}")
        label_issues.extend(run_issues)
    else:
        print(f"  All {len(sample_subjects)} subjects: T1=2, T2=3 confirmed")

if not label_issues:
    print(f"\nLabel mapping confirmed correct across all checked subjects and runs")
else:
    print(f"\nWarning: {len(label_issues)} label issues found, investigate before proceeding")

Checking label mapping across all 106 subjects and 3 runs...
Expected: T1=2 (left fist), T2=3 (right fist)

Run R04:
  All 10 subjects: T1=2, T2=3 confirmed
Run R08:
  All 10 subjects: T1=2, T2=3 confirmed
Run R12:
  All 10 subjects: T1=2, T2=3 confirmed

Label mapping confirmed correct across all checked subjects and runs


#### checking if both classes present in all 106 subjects

In [16]:
missing_class = []

for subject_id in valid_subjects:
    data = np.load(processed_path / f'{subject_id}_processed.npz')
    labels = data['labels']
    unique_labels = np.unique(labels)

    if 2 not in unique_labels or 3 not in unique_labels:
        missing_class.append(f"{subject_id}: labels {unique_labels}")

if not missing_class:
    print("All 106 subjects have both T1 and T2 trials confirmed")
else:
    print(f"Subjects missing a class: {missing_class}")

All 106 subjects have both T1 and T2 trials confirmed


### data range check across all 106 subjects

In [17]:
# after z-score normalisation, a check to identify any extreme values.
# two tier data range check across all 106 subjects
# tier 1: absolute check, catches catastrophic pipeline failures
# tier 2: percentage check, catches widespread residual artifact

absolute_limit = 15
pct_limit_std = 5
pct_limit_threshold = 0.1

absolute_flags = []
percentage_flags = []

for subject_id in valid_subjects:
    data = np.load(processed_path / f'{subject_id}_processed.npz')
    d = data['data']

    # tier 1: absolute check
    if np.abs(d).max() > absolute_limit:
        absolute_flags.append(
            f"{subject_id}: min {d.min():.2f} max {d.max():.2f}"
        )

    # tier 2: percentage check
    pct_extreme = (np.abs(d) > pct_limit_std).mean() * 100
    if pct_extreme > pct_limit_threshold:
        percentage_flags.append(
            f"{subject_id}: {pct_extreme:.4f}% values beyond {pct_limit_std} std"
        )

print(f"Tier 1: absolute check (flag if any value exceeds {absolute_limit})")
if not absolute_flags:
    print(f"  No subjects flagged")
else:
    for flag in absolute_flags:
        print(f"  {flag}")

print(f"\nTier 2: percentage check (flag if >{pct_limit_threshold}% values exceed {pct_limit_std} std)")
if not percentage_flags:
    print(f"  No subjects flagged")
else:
    for flag in percentage_flags:
        print(f"  {flag}")

print(f"\nTier 1 flags: {len(absolute_flags)} subjects, investigate pipeline if flagged")
print(f"Tier 2 flags: {len(percentage_flags)} subjects, monitor during model evaluation")

Tier 1: absolute check (flag if any value exceeds 15)
  S051: min -16.43 max 15.81
  S063: min -13.22 max 33.41
  S086: min -22.27 max 20.88

Tier 2: percentage check (flag if >0.1% values exceed 5 std)
  S086: 0.1892% values beyond 5 std

Tier 1 flags: 3 subjects, investigate pipeline if flagged
Tier 2 flags: 1 subjects, monitor during model evaluation


In [18]:
# check flagged subjects in more detail, to check whether extreme values are isolated spikes or widespread
# check on tier 1 flagged subjects

for subject_id in ['S051', 'S063', 'S086']:
    data = np.load(processed_path / f'{subject_id}_processed.npz')
    d = data['data']
    pct_above_15 = (np.abs(d) > 15).mean() * 100
    pct_above_5 = (np.abs(d) > 5).mean() * 100
    print(f"{subject_id}:")
    print(f"  min: {d.min():.2f}, max: {d.max():.2f}")
    print(f"  % above 15 std: {pct_above_15:.6f}%")
    print(f"  % above 5 std: {pct_above_5:.4f}%")

S051:
  min: -16.43, max: 15.81
  % above 15 std: 0.001184%
  % above 5 std: 0.0207%
S063:
  min: -13.22, max: 33.41
  % above 15 std: 0.002894%
  % above 5 std: 0.0284%
S086:
  min: -22.27, max: 20.88
  % above 15 std: 0.013310%
  % above 5 std: 0.1892%


S051, S063: isolated single spikes, less than 0.003% of values exceed 15 std.
Not a preprocessing error.

S086: flagged in both tiers, 0.1892% of values exceed 5 std.
Most residual artifact of any subject. It will be monitored during LOSO evaluation,
specifically checking whether S086 as a test subject produces unusually low accuracy
and whether including S086 in training affects other subjects.

### channel and shape consistency across all 106 subjects

In [19]:
# all subjects must have identical channel order and 480 timepoints
# mismatch causes errors during model training

shape_issues = []
channel_issues = []

expected_shape_1 = 8
expected_shape_2 = 480
expected_channels = list(selected_channels)

for subject_id in valid_subjects:
    data = np.load(processed_path / f'{subject_id}_processed.npz',
                   allow_pickle=True)
    shape = data['data'].shape
    channels = list(data['selected_channels'])

    if shape[1] != expected_shape_1 or shape[2] != expected_shape_2:
        shape_issues.append(f"{subject_id}: {shape}")

    if channels != expected_channels:
        channel_issues.append(f"{subject_id}: {channels}")

print(f"Shape consistency check:")
if not shape_issues:
    print(f"  All 106 subjects: 8 channels x 480 timepoints confirmed")
else:
    print(f"  Issues: {shape_issues}")

print(f"\nChannel consistency check:")
if not channel_issues:
    print(f"  All 106 subjects: channel order confirmed")
else:
    print(f"  Issues: {channel_issues}")

Shape consistency check:
  All 106 subjects: 8 channels x 480 timepoints confirmed

Channel consistency check:
  All 106 subjects: channel order confirmed


### Checks for rework

In [22]:
# FC4 lateralisation check across all 106 subjects
# recomputing since FC4 was excluded from final processed files

fc4_ratios = []

for subject_id in valid_subjects:
    raw_list = []
    for run in ['R04', 'R08', 'R12']:
        edf_path = data_path / subject_id / f"{subject_id}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw_s.notch_filter(freqs=60, verbose=False)
        raw_s.filter(l_freq=8.0, h_freq=30.0, method='fir',
                    fir_design='firwin', verbose=False)
        raw_s.set_eeg_reference('average', projection=False, verbose=False)
        raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(raw_list, verbose=False)
    raw_concat = rename_channels_standard(raw_concat)

    if 'FC4' not in raw_concat.ch_names:
        continue

    events, _ = mne.events_from_annotations(raw_concat, verbose=False)
    epochs = mne.Epochs(raw_concat, events, event_id={'T1': 2, 'T2': 3},
                        tmin=0, tmax=3, baseline=None, preload=True, verbose=False)

    fc4_data = epochs.get_data(picks=['FC4'])
    labels = epochs.events[:, -1]

    left_var = fc4_data[labels == 2].var()
    right_var = fc4_data[labels == 3].var()
    ratio = right_var / left_var if left_var > 0 else np.nan

    fc4_ratios.append({'subject_id': subject_id, 'fc4_ratio': ratio})

fc4_df = pd.DataFrame(fc4_ratios)
print(f"Subjects processed: {len(fc4_df)}")
print(f"Mean FC4 ratio: {fc4_df['fc4_ratio'].mean():.4f}")
print(f"Std FC4 ratio: {fc4_df['fc4_ratio'].std():.4f}")
print(f"Subjects with ratio < 1.0 (correct direction): {(fc4_df['fc4_ratio'] < 1.0).sum()}/{len(fc4_df)}")
print(f"Subjects with ratio > 1.0 (wrong direction): {(fc4_df['fc4_ratio'] > 1.0).sum()}/{len(fc4_df)}")

Subjects processed: 106
Mean FC4 ratio: 1.0535
Std FC4 ratio: 0.1848
Subjects with ratio < 1.0 (correct direction): 39/106
Subjects with ratio > 1.0 (wrong direction): 67/106


### refrencing check

In [23]:
# full dataset referencing recheck
# common average vs linked mastoid vs laplacian
# lateral separation via C3/C4 variance ratio across T1/T2

referencing_results = []

montage = mne.channels.make_standard_montage('standard_1020')
montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

def fix_ch(name):
    upper = name.upper()
    return montage_lookup[upper] if upper in montage_lookup else name

for subject_id in valid_subjects:
    raw_list_avg = []
    raw_list_mastoid = []
    raw_list_lap = []

    for run in ['R04', 'R08', 'R12']:
        edf_path = data_path / subject_id / f"{subject_id}{run}.edf"
        raw_base = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw_base.notch_filter(freqs=60, verbose=False)
        raw_base.filter(l_freq=8.0, h_freq=30.0, method='fir',
                        fir_design='firwin', verbose=False)
        raw_base = rename_channels_standard(raw_base)

        # common average
        raw_avg = raw_base.copy()
        raw_avg.set_eeg_reference('average', projection=False, verbose=False)
        raw_list_avg.append(raw_avg)

        # linked mastoid (using TP9/TP10 or closest available as proxy)
        raw_mastoid = raw_base.copy()
        mastoid_chs = [ch for ch in ['TP9', 'TP10', 'M1', 'M2']
                      if ch in raw_mastoid.ch_names]
        if len(mastoid_chs) >= 2:
            raw_mastoid.set_eeg_reference(mastoid_chs, projection=False, verbose=False)
        else:
            raw_mastoid.set_eeg_reference('average', projection=False, verbose=False)
        raw_list_mastoid.append(raw_mastoid)

        # laplacian
        raw_lap = raw_base.copy()
        cap_fix = {ch: fix_ch(ch) for ch in raw_lap.ch_names}
        raw_lap.rename_channels(cap_fix)
        raw_lap.set_montage(montage, on_missing='ignore', verbose=False)
        try:
            raw_lap = mne.preprocessing.compute_current_source_density(raw_lap, verbose=False)
        except Exception:
            pass
        raw_list_lap.append(raw_lap)

    try:
        for ref_name, raw_list in [('avg', raw_list_avg),
                                     ('mastoid', raw_list_mastoid),
                                     ('laplacian', raw_list_lap)]:
            raw_concat = mne.concatenate_raws(raw_list, verbose=False)
            events, _ = mne.events_from_annotations(raw_concat, verbose=False)
            epochs = mne.Epochs(raw_concat, events, event_id={'T1': 2, 'T2': 3},
                                tmin=0, tmax=3, baseline=None, preload=True, verbose=False)

            if 'C3' not in epochs.ch_names or 'C4' not in epochs.ch_names:
                continue

            c3_data = epochs.get_data(picks=['C3'])
            c4_data = epochs.get_data(picks=['C4'])
            labels = epochs.events[:, -1]

            c3_t1 = c3_data[labels == 2].var()
            c3_t2 = c3_data[labels == 3].var()
            c4_t1 = c4_data[labels == 2].var()
            c4_t2 = c4_data[labels == 3].var()

            t1_ratio = c3_t1 / c4_t1 if c4_t1 > 0 else np.nan
            t2_ratio = c3_t2 / c4_t2 if c4_t2 > 0 else np.nan

            referencing_results.append({
                'subject_id': subject_id,
                'reference': ref_name,
                't1_ratio': t1_ratio,
                't2_ratio': t2_ratio,
                'diff': abs(t1_ratio - t2_ratio) if not (np.isnan(t1_ratio) or np.isnan(t2_ratio)) else np.nan
            })
    except Exception as e:
        print(f"Error for {subject_id}: {e}")
        continue

ref_df = pd.DataFrame(referencing_results)
print(f"Total records: {len(ref_df)}")

summary = ref_df.groupby('reference').agg({
    't1_ratio': 'mean',
    't2_ratio': 'mean',
    'diff': 'mean'
})
print(summary)

print("\nMean separation difference by method (higher is better):")
for ref in ['avg', 'mastoid', 'laplacian']:
    subset = ref_df[ref_df['reference'] == ref]
    print(f"  {ref}: {subset['diff'].mean():.4f} (n={len(subset)})")

Total records: 318
           t1_ratio  t2_ratio      diff
reference                              
avg        1.163330  0.949115  0.237815
laplacian  1.384147  1.102238  0.296601
mastoid    1.163330  0.949115  0.237815

Mean separation difference by method (higher is better):
  avg: 0.2378 (n=106)
  mastoid: 0.2378 (n=106)
  laplacian: 0.2966 (n=106)
